In [1]:
import osmnx as ox
import networkx as nx 
import folium
import sklearn
# print("osmnx version:", ox.__version__)

In [2]:
# 出発地点・到着地点の設定 (緯度, 経度)
origin_point      = (34.30202568645756, 134.04677822902838)
destination_point = (34.29323159568179, 134.06659364232908)

# 2点の中心座標
center_lat = (origin_point[0] + destination_point[0]) / 2
center_lon = (origin_point[1] + destination_point[1]) / 2

# 道路ネットワークを取得（徒歩 / 半径2000m）
G = ox.graph_from_point(
    (center_lat, center_lon),
    dist=3000,
    network_type='walk'
)

print("道路ネットワーク取得完了")
print(f"ノード数: {len(G.nodes)}, エッジ数: {len(G.edges)}")

道路ネットワーク取得完了
ノード数: 14374, エッジ数: 38342


In [3]:
# 出発・到着地点に最も近いノードを取得
origin_node      = ox.distance.nearest_nodes(G, origin_point[1], origin_point[0])
destination_node = ox.distance.nearest_nodes(G, destination_point[1], destination_point[0])

# ダイクストラ法で最短経路を探索
route = nx.shortest_path(
    G,
    source=origin_node,
    target=destination_node,
    weight='length',
    method='dijkstra'
)

# 総距離を計算
route_length = nx.shortest_path_length(
    G,
    source=origin_node,
    target=destination_node,
    weight='length',
    method='dijkstra'
)

print(f"経路のノード数: {len(route)}")
print(f"総距離: {route_length:.0f} m  ({route_length/1000:.2f} km)")

経路のノード数: 64
総距離: 2619 m  (2.62 km)


In [4]:
# 経路のノードから座標リストを作成 (foliumはlatlon順)
route_coords = [
    (G.nodes[node]['y'], G.nodes[node]['x'])
    for node in route
]

# foliumマップを作成
m = folium.Map(location=[center_lat, center_lon], zoom_start=15)

# 経路を赤いラインで描画
folium.PolyLine(
    locations=route_coords,
    color='red',
    weight=5,
    opacity=0.8,
    tooltip=f'最短経路: {route_length:.0f}m'
).add_to(m)

# 出発地点マーカー（青）
folium.Marker(
    location=origin_point,
    popup=folium.Popup('出発地点', parse_html=True),
    tooltip='出発地点',
    icon=folium.Icon(color='blue', icon='play')
).add_to(m)

# 到着地点マーカー（緑）
folium.Marker(
    location=destination_point,
    popup=folium.Popup('到着地点', parse_html=True),
    tooltip='到着地点',
    icon=folium.Icon(color='green', icon='flag')
).add_to(m)

# 距離情報をマップ上に表示
folium.Marker(
    location=[center_lat, center_lon],
    icon=folium.DivIcon(
        html=f'<div style="font-size:13px; color:black; background:white; padding:4px 8px; '
             f'border-radius:4px; border:1px solid gray;">'
             f'総距離: {route_length:.0f}m ({route_length/1000:.2f}km)</div>'
    )
).add_to(m)

# HTMLファイルとして保存
m.save('route_map.html')
print("地図を route_map.html に保存しました")

# Jupyter上で表示
m

地図を route_map.html に保存しました
